# NextRun — retrain the court and ball/rim models (Colab, GPU)

**Runtime → Change runtime type → T4 GPU**, and add the secret **`ROBOFLOW_API_KEY`** (key icon on the left).
Each section: build the dataset → score the *current* model → train (time-capped, so a free session can't
expire mid-run) → score the *new* model on the same test images → install only if it is better.

| Model | Data | Held-out check |
|---|---|---|
| Court (24 keypoints) | NBA broadcast (850 frames, HF) · TrackID3x3 outdoor 3x3 (12 recordings) and indoor gym (42 clips), auto-labelled through their calibrations · Rishabh's HS sets (Roboflow, if listed) | TrackID3x3 **drone** court: a court and viewpoint never trained on |
| Ball + rim | 6 Roboflow + 3 Hugging Face sets labelling both ball and rim (~10.8k frames) | test split per set **and** your own footage (ball found per frame) |

Splits are by video / game clip, so near-duplicate frames never land in both train and test.

In [ ]:
!git clone -b model_merge https://github.com/aisutd/nextrun.git /content/nextrun || (cd /content/nextrun && git pull)
%cd /content/nextrun
!pip install -q -e ".[detect,train]" gdown
import os, json, shutil
from google.colab import userdata
os.environ["ROBOFLOW_API_KEY"] = userdata.get("ROBOFLOW_API_KEY")  # key icon -> secret ROBOFLOW_API_KEY
os.makedirs("results", exist_ok=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 0. TrackID3x3 videos (public Google Drive, ~18 GB; a few minutes on Colab)
Outdoor: 12 recordings of one fixed iPhone (the camera moves 10-23 px between them; the builder re-registers
each). Indoor: 42 short clips of a fixed gym camera. Drone 1-1: a different outdoor court, test only.

In [ ]:
OUTDOOR = {"IMG_0104": "1B4WgkrIrPaIe-kIRd5Kq8XouUmezolL_", "IMG_0105": "1JmRa6HJp-06pyxHp11G0YOb05tf1wtcC",
           "IMG_0106": "1TGBRoqUcUqZUL33tI1up9OobLtOCQNhw", "IMG_0107": "12Ge8yz1W2BKnrxpX-TM1_ZttArVLnMc0",
           "IMG_0108": "13j5W53tJqPYpTKSFI_zyi3LMqZVy-EFH", "IMG_0109": "134s82zCspw8zdSYBa21__fHLnIXSialZ",
           "IMG_0110": "1H93_doyUuuXQg1SWR-g5FQRXjjUf5bSu", "IMG_0111": "1FJeSP6gSITTjyA6IPJWKlfVDu4RlOWfF",
           "IMG_0112": "1Oc1YlDw4eUfXV-iga4Z_dbkGhVZ-67Pp", "IMG_0113": "1gL63ham7IZM6SLGVTCu-wcJwc3weJ9eL",
           "IMG_0114": "13sIrq7X5xn2Fp-_kQS_mRpI7NoH5PD6M", "IMG_0115": "1e3-gRZ4luCu_2XwiZpuHQpGUHqtostB1"}
T = "data/raw/trackid3x3/videos"
os.makedirs(f"{T}/outdoor", exist_ok=True); os.makedirs(f"{T}/drone", exist_ok=True)
for name, fid in OUTDOOR.items():
    if not os.path.exists(f"{T}/outdoor/{name}.MOV"):
        !gdown -q {fid} -O {T}/outdoor/{name}.MOV
if not os.path.exists(f"{T}/drone/1-1.mov"):
    !gdown -q 1Xd5dcy3N5SyvdQ887lownjvL38fLl5JD -O {T}/drone/1-1.mov
if not os.path.isdir(f"{T}/indoor"):
    !gdown -q --folder 1q0L_Mmf5cwb-Yv51NTjgQcF12Is0fYCB -O {T}/indoor
!ls {T}/outdoor | wc -l; ls {T}/indoor | wc -l; ls {T}/drone

## 1. Court model
Add Rishabh's 24-point sets to `COURT_ROBOFLOW` (`workspace/project/version`) if you have them: without them
the high-school-gym domain can't be measured, and the model may drift away from it.

In [ ]:
COURT_ROBOFLOW = [
    # "workspace/project/3",   # Rishabh: corner view, high-school gyms
]
rf = ("--roboflow " + " ".join(COURT_ROBOFLOW)) if COURT_ROBOFLOW else ""
!python src/scripts/build_court_dataset.py --out data/court24 --nba-hf {rf} \
    --venue trackid3x3_outdoor "{T}/outdoor/IMG_0114.MOV,{T}/outdoor/*.MOV" --venue-ref-frame trackid3x3_outdoor 900 \
    --venue trackid3x3_indoor "{T}/indoor/*.mp4" \
    --venue trackid3x3_drone {T}/drone/1-1.mov --venue-ref-frame trackid3x3_drone 300 --test-venue trackid3x3_drone \
    --every 300
# score the CURRENT court model on every domain's test split
!python src/scripts/eval_court.py --data data/court24 --split test | tee results/court_before.txt

In [ ]:
# fine-tune from the current model (keeps what it learned on high-school gyms) at 1280 px; capped at 1.5 h (~T4)
!python src/scripts/train.py court --data data/court24/data.yaml --model models/court_model.pt --imgsz 1280 --epochs 150 --hours 1.5 --batch 16 --no-install --name court_v2
!python src/scripts/eval_court.py --data data/court24 --split test --weights results/runs/court/court_v2/weights/best.pt | tee results/court_after.txt

In [ ]:
# optional side run: a bigger / newer architecture from COCO-pretrained weights ("" to skip). Compare its drone-court
# numbers with court_v2 above; it starts without the high-school-gym training, so only switch if it clearly wins.
COURT_ALT = "yolo26m-pose.pt"   # or "yolo11m-pose.pt"
if COURT_ALT:
    !python src/scripts/train.py court --data data/court24/data.yaml --model {COURT_ALT} --imgsz 1280 --epochs 150 --hours 1.5 --batch 16 --no-install --name court_alt
    !python src/scripts/eval_court.py --data data/court24 --split test --weights results/runs/court/court_alt/weights/best.pt | tee results/court_alt_after.txt

In [ ]:
# install only if better (lower median error, same-or-better found rate) on EVERY domain, drone court included
before = json.loads(open("results/court_before.txt").read().splitlines()[-1])
after = json.loads(open("results/court_after.txt").read().splitlines()[-1])
nan = lambda v: v != v
for s in before:
    print(f"{s:36s} found {before[s]['found_rate']:.2f} -> {after[s]['found_rate']:.2f}   "
          f"median ft {before[s]['err_ft_median']} -> {after[s]['err_ft_median']}")
better = all(after[s]["found_rate"] >= before[s]["found_rate"] - 0.02 and
             (nan(before[s]["err_ft_median"]) or after[s]["err_ft_median"] <= before[s]["err_ft_median"]) for s in before)
print("INSTALL" if better else "NOT better everywhere - keeping the current model (new weights still downloaded below)")
if better:
    shutil.copyfile("models/court_model.pt", "models/court_model.prev.pt")
    shutil.copyfile("results/runs/court/court_v2/weights/best.pt", "models/court_model.pt")

## 2. Ball + rim model
Rim boxes are rim + net; sets that label only one of ball / rim are skipped. Add your own labelled frames
(Roboflow export) to `BALL_ROBOFLOW` — frames from **your** cameras help most.

In [ ]:
BALL_ROBOFLOW = ["suggested"]   # + "your-workspace/your-project/1"
!python src/scripts/build_ball_rim_dataset.py --out data/ball_rim --hf suggested --roboflow {" ".join(BALL_ROBOFLOW)}
!python src/scripts/eval_ball_rim.py --data data/ball_rim --split test | tee results/ball_before.txt

In [ ]:
# YOLO26 targets small objects (our ball problem) and is NMS-free. s fits ~40 epochs at 1024 px in 3.5 h on a T4;
# "yolo26m.pt" is stronger but gets ~half the epochs in the same time; "yolo11s.pt" is the previous generation.
BALL_MODEL = "yolo26s.pt"
!python src/scripts/train.py ball --data data/ball_rim/data.yaml --model {BALL_MODEL} --imgsz 1024 --epochs 80 --hours 3.5 --batch 16 --no-install --name ball_v2
!python src/scripts/eval_ball_rim.py --data data/ball_rim --split test --weights results/runs/ball/ball_v2/weights/best.pt | tee results/ball_after.txt
# the real check: our own footage (labelled sets are mostly broadcast; a CPU run that beat them still missed balls here)
!python src/scripts/check_ball_footage.py {T}/outdoor/IMG_0105.MOV {T}/outdoor/IMG_0110.MOV {T}/indoor/basket_S3T2_pre.mp4 {T}/indoor/basket_S5T4_post.mp4 \
    --weights models/ball_hoop_model.pt results/runs/ball/ball_v2/weights/best.pt --save results/ball_disagreements | tee results/ball_footage.txt

In [ ]:
before = json.loads(open("results/ball_before.txt").read().splitlines()[-1])
after = json.loads(open("results/ball_after.txt").read().splitlines()[-1])
foot = json.loads(open("results/ball_footage.txt").read().splitlines()[-1])
for s in before:
    for c in ("ball", "rim"):
        if c in before[s] and c in after.get(s, {}):
            print(f"{s:40s} {c:4s} recall {before[s][c]['R']} -> {after[s][c]['R']}   mAP50 {before[s][c]['mAP50']} -> {after[s][c]['mAP50']}")
mean = lambda d, c: sum(d[s][c]["mAP50"] for s in d if c in d[s]) / max(1, sum(c in d[s] for s in d))
old, new = foot["models/ball_hoop_model.pt"], foot["results/runs/ball/ball_v2/weights/best.pt"]
print(f"mean ball mAP50 {mean(before, 'ball'):.3f} -> {mean(after, 'ball'):.3f}, rim {mean(before, 'rim'):.3f} -> {mean(after, 'rim'):.3f}")
print(f"own footage: ball found {old['ball_found']} -> {new['ball_found']} of {new['frames']}, rim {old['rim_found']} -> {new['rim_found']}")
better = (mean(after, "ball") > mean(before, "ball") and mean(after, "rim") >= mean(before, "rim") - 0.01
          and new["ball_found"] >= old["ball_found"] and new["rim_found"] >= old["rim_found"])
print("INSTALL" if better else "NOT better - keeping the current model (look at results/ball_disagreements; weights still downloaded below)")
if better:
    shutil.copyfile("models/ball_hoop_model.pt", "models/ball_hoop_model.prev.pt")
    shutil.copyfile("results/runs/ball/ball_v2/weights/best.pt", "models/ball_hoop_model.pt")

## 3. Keep the results
Downloads the new weights (installed or not) and the reports; copy them into the repo (`models/`, `results/`).

In [ ]:
!cp results/runs/court/court_v2/weights/best.pt results/court_v2_best.pt 2>/dev/null; cp results/runs/court/court_alt/weights/best.pt results/court_alt_best.pt 2>/dev/null; cp results/runs/ball/ball_v2/weights/best.pt results/ball_v2_best.pt 2>/dev/null
!cd results && zip -qr ../nextrun_colab_results.zip . -x "runs/*"
from google.colab import files
files.download("nextrun_colab_results.zip")